## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 25 samples and 11825 columns.
First few rows of the data:


,os.time,os.event,pfs.time,pfs.event,samples,sample.timepoint,ajcc.stage,tumor.response,type.of.lesion,type.of.primary,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
0,92.9,0,48.1,1,MM909_01_1,Before TILs,M1c,-100.00,LN,unknown,...,3.14586,2.20174,2.70930,0.82517,1.36987,2.76248,2.55180,5.53388,2.56353,3.15532
1,6.9,1,2.0,1,MM909_02_1,Before TILs,M1b,10.94,LN,skin,...,3.36122,3.40559,2.97088,1.27695,1.29983,2.69095,2.45870,5.42459,1.80400,3.68111
2,11.4,1,3.8,1,MM909_03_1,Before TILs,M1c,11.92,SC,unknown,...,3.29989,2.79518,3.91229,1.19076,2.07418,2.84976,3.43125,5.08218,2.69648,4.74777
3,4.6,1,2.3,1,MM909_06_1,Before TILs,M1c,-7.14,LN,skin,...,3.00165,3.22395,4.48321,0.38875,0.91491,1.91299,2.96821,5.00434,0.54810,3.79871
4,71.5,0,13.1,1,MM909_11_1,Before TILs,M1a,-100.00,LN,skin,...,2.73279,3.54445,3.98488,0.71845,1.15190,2.64073,2.92381,5.07525,2.02028,3.90744


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'samples'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
